In [1]:
spark.sql("USE cryptodb_trusted")
spark.sql("""
    SELECT Name, 
           ROUND(AVG((High-Low)/Close*100), 3) AS volatilidad_pct
    FROM crypto_prices
    GROUP BY Name
    ORDER BY volatilidad_pct DESC
""").show()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1779309328797_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+---------------+
|     Name|volatilidad_pct|
+---------+---------------+
|   Solana|         13.676|
|Chainlink|          11.08|
|  Cardano|           9.05|
| Ethereum|          7.054|
|  Bitcoin|          5.304|
+---------+---------------+

In [2]:
spark.sql("""
    SELECT Name, Year,
           ROUND(((MAX(Close) - MIN(Close)) / MIN(Close)) * 100, 2) AS crecimiento_pct
    FROM crypto_prices
    WHERE Year BETWEEN 2020 AND 2024
    GROUP BY Name, Year
    ORDER BY Year, crecimiento_pct DESC
""").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+----+---------------+
|     Name|Year|crecimiento_pct|
+---------+----+---------------+
|Chainlink|2020|         996.91|
|   Solana|2020|         828.06|
|  Cardano|2020|         701.99|
| Ethereum|2020|         579.55|
|  Bitcoin|2020|         483.44|
|   Solana|2021|        3007.42|
|  Cardano|2021|        1216.86|
| Ethereum|2021|         470.77|
|Chainlink|2021|         339.66|
|  Bitcoin|2021|         116.19|
+---------+----+---------------+

In [3]:
spark.sql("""
    SELECT Name, Year, Month,
           ROUND(AVG(Close), 2) AS precio_promedio
    FROM crypto_prices
    WHERE Name IN ('Bitcoin', 'Ethereum', 'Solana')
    AND Year BETWEEN 2020 AND 2024
    GROUP BY Name, Year, Month
    ORDER BY Name, Year, Month
""").show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+----+-----+---------------+
|    Name|Year|Month|precio_promedio|
+--------+----+-----+---------------+
| Bitcoin|2020|    1|        8389.27|
| Bitcoin|2020|    2|        9630.72|
| Bitcoin|2020|    3|        6871.02|
| Bitcoin|2020|    4|        7224.48|
| Bitcoin|2020|    5|        9263.15|
| Bitcoin|2020|    6|        9489.23|
| Bitcoin|2020|    7|         9589.9|
| Bitcoin|2020|    8|       11652.39|
| Bitcoin|2020|    9|       10660.28|
| Bitcoin|2020|   10|       11886.98|
| Bitcoin|2020|   11|       16645.76|
| Bitcoin|2020|   12|       21983.14|
| Bitcoin|2021|    1|       34761.65|
| Bitcoin|2021|    2|        46306.8|
| Bitcoin|2021|    3|       54998.01|
| Bitcoin|2021|    4|       57206.72|
| Bitcoin|2021|    5|       46443.29|
| Bitcoin|2021|    6|       35845.15|
| Bitcoin|2021|    7|       34234.45|
|Ethereum|2020|    1|         156.97|
+--------+----+-----+---------------+
only showing top 20 rows

In [4]:
spark.sql("""
    SELECT Month,
           ROUND(AVG(daily_return), 4) AS rendimiento_pct,
           COUNT(*) AS dias_muestra
    FROM crypto_prices
    GROUP BY Month
    ORDER BY Month
""").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+---------------+------------+
|Month|rendimiento_pct|dias_muestra|
+-----+---------------+------------+
| NULL|           NULL|           5|
|    1|         1.0286|         589|
|    2|         0.9156|         536|
|    3|        -0.0365|         589|
|    4|         1.1847|         590|
|    5|         0.6553|         620|
|    6|         0.2032|         600|
|    7|         0.4544|         495|
|    8|         0.4583|         465|
|    9|         -0.116|         460|
|   10|         0.1475|         526|
|   11|         0.5527|         510|
|   12|         0.9674|         527|
+-----+---------------+------------+

In [6]:
from pyspark.sql.functions import col

for coin in ['Bitcoin', 'Ethereum', 'Solana', 'Cardano', 'Chainlink']:
    df_coin = spark.sql(f"""
        SELECT CAST(Volume AS DOUBLE) AS Volume, 
               CAST(Close AS DOUBLE) AS Close 
        FROM crypto_prices 
        WHERE Name = '{coin}'
    """)
    corr = df_coin.stat.corr('Volume', 'Close')
    print(f"{coin}: r = {corr:.4f}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Bitcoin: r = 0.7397
Ethereum: r = 0.7353
Solana: r = 0.8062
Cardano: r = 0.7566
Chainlink: r = 0.2239